In [ ]:
import h5py
import plotly.graph_objects as go
import pandas as pd
from plotly.colors import sample_colorscale
import numpy as np
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [ ]:
def load_h5(h5_path: str) -> dict:
    """
    Safely loads simulation data from legacy or new HDF5 structures.
    """
    print(f"Opening {h5_path}...")

    data = []
    stress = []
    global_data = {}

    with h5py.File(h5_path, "r") as f:
        # Isolate heavy Global Mesh data
        quad_coords = (
            f["quad_coordinates"][()].flatten() if "quad_coordinates" in f else None
        )
        nodes_coords = (
            f["nodes_coordinates"][()].flatten() if "nodes_coordinates" in f else None
        )
        sim_time = f.attrs.get("simulation_time", 0.0)

        # Lazy iteration
        for name in f:
            item = f[name]

            # Process step groups
            if isinstance(item, h5py.Group) and name.startswith("step_"):
                try:
                    step_idx = int(name.split("_")[1])
                except ValueError:
                    continue

                step_meta = {"step": step_idx}

                # Helper to route scalars to the dict and reject heavy arrays
                def process_item(source: dict) -> None:
                    for k, v in source.items():
                        # Unpack hdf5 dataset to numpy array
                        val = v[()] if isinstance(v, h5py.Dataset) else v
                        if isinstance(val, np.ndarray):
                            if (
                                k not in ["fragment_mass", "fragment_velocity"]
                                and val.size > 100
                            ):
                                continue
                        if isinstance(val, (list, tuple)) and len(val) > 100:
                            continue
                        step_meta[k] = val

                process_item(item.attrs)
                process_item(
                    {k: item[k] for k in item if isinstance(item[k], h5py.Dataset)}
                )

                data.append(step_meta)

                # Isolate stress
                stress_array = None
                if "stress" in item:
                    stress_array = item["stress"][()]
                elif "stress" in item.attrs:
                    stress_array = item.attrs["stress"]

                if stress_array is not None:
                    stress.append((step_idx, step_meta.get("time", None), stress_array))

            # Process other global data
            elif isinstance(item, h5py.Dataset) and name not in [
                "quad_coordinates",
                "nodes_coordinates",
            ]:
                global_data[name] = item[()]

    # Post-Processing: Build the clean Pandas DataFrame
    df = pd.DataFrame(data)
    if not df.empty and "step" in df.columns:
        df = df.set_index("step").sort_index()

    # Post-Processing: Build the Plotly-ready Space-Time matrices
    if stress:
        stress.sort(key=lambda x: x[0])  # Guarantee monotonic time
        stress_times = np.array([x[1] for x in stress])
        stress_matrix = np.vstack([x[2] for x in stress])
    else:
        stress_times, stress_matrix = None, None

    print(f"Loaded {len(df)} steps. Simulation Time: {sim_time:.2f}s")

    return {
        "df": df,
        "stress_times": stress_times,
        "stress_matrix": stress_matrix,
        "quad_coords": quad_coords,
        "nodes_coords": nodes_coords,
        "global_data": global_data,
    }


# Layout for plots
def get_layout() -> go.Layout:
    layout = go.Layout(
        xaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            tickfont=dict(size=12),
        ),
        yaxis=dict(
            showgrid=True,
            showline=True,
            linewidth=1,
            linecolor="black",
            mirror=True,
            zeroline=False,
            ticks="inside",
            exponentformat="power",
            tickfont=dict(size=12),
        ),
        font=dict(family="Latin-Modern", size=12, color="Black"),
        legend=dict(
            x=0.97,
            y=1.2,
            bgcolor="rgba(255, 255, 255, 0.8)",
            bordercolor="black",
            borderwidth=1,
            orientation="h",
            xanchor="right",
            yanchor="top",
        ),
        width=600,
        height=500,
        showlegend=True,
        template="plotly_white",
    )
    return layout

In [ ]:
# Examples from the README examples
run_id = "output/test_penalty"
run_id = "output/test_nsn"
run_id = "output/test_impact_cir0.1"

# Example from the reproduce.sh script
run_id = "output/test_reproduce"

In [ ]:
# Load data
run_path = f"../../{run_id}/data.h5"
data_dict = load_h5(run_path)

# Extract data
df = data_dict["df"]
stress_times = data_dict["stress_times"]
stress_matrix = data_dict["stress_matrix"]
quad_coords = data_dict["quad_coords"]

# Colors for plots
colorscale = "Spectral_r"
samples = [0.0, 0.1, 0.25, 0.35]  # , 0.6]
colors = sample_colorscale(colorscale, samples)
colors.append("lightgrey")
colors.append("grey")
fillcolors = [c.replace("rgb(", "rgba(").replace(")", ",0.7)") for c in colors]

In [ ]:
### Plots: energy balance, energy variation, stacked energy, number of fragments ###

write = False

# Algorithmic and mechanical energy balance vs. Time
fig_balance = go.Figure()
fig_balance.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["algorithmic_energy_balance"] - df["algorithmic_energy_balance"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{E}^{\rm alg}$",
        line=dict(color="black"),
    )
)
fig_balance.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["mechanical_energy_balance"] - df["mechanical_energy_balance"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{E}^{\rm mech}$",
        line=dict(color="red"),
    )
)
fig_balance.update_layout(
    get_layout(),
    title="Energy balance vs. Time",
    xaxis_title="Time (s)",
    yaxis_title=r"$\Delta\mathcal{E}$ (J)",
    legend=dict(y=0.97),
)
fig_balance.show()
if write:
    fig_balance.write_image(
        f"../../output/{run_id}/energy_balance_vs_time_2.pdf",
        width=400,
        height=300,
        scale=2,
    )


# Energy variation vs. Time
fig_energy = go.Figure()
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["kinetic_energy"] - df["kinetic_energy"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{K}$",
        line=dict(color=colors[0]),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["potential_energy"] - df["potential_energy"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{U}$",
        line=dict(color=colors[1]),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["reversible_energy"] - df["reversible_energy"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{R}$",
        line=dict(color=colors[2]),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["contact_energy"] - df["contact_energy"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{C}_{\rm rev}$",
        line=dict(color=colors[3]),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["dissipated_energy"] - df["dissipated_energy"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{G}$",
        line=dict(color=colors[4]),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["contact_dissipation"] - df["contact_dissipation"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{C}_{\rm dis}$",
        line=dict(color=colors[5]),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=-(df["external_work"] - df["external_work"].iloc[0]),
        mode="lines",
        name=r"$\mathcal{W}^{\rm ext}$",
        line=dict(color="red"),
    )
)
fig_energy.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["mechanical_energy_balance"] - df["mechanical_energy_balance"].iloc[0],
        mode="lines",
        name=r"$\Delta\mathcal{E}^{\rm mech}$",
        line=dict(color="black"),
    )
)

fig_energy.update_layout(
    get_layout(),
    title="Energy variation vs. Time",
    xaxis_title="Time (s)",
    yaxis_title=r"$\Delta\mathcal{E}$ (J)",
    legend=dict(y=1.2),
)
fig_energy.show()
if write:
    fig_energy.write_image(
        f"../../output/{run_id}/energy_vs_time.pdf", width=400, height=400, scale=2
    )

# Stacked Energy vs. Time
ekin_stacked = df["kinetic_energy"]
epot_stacked = ekin_stacked + df["potential_energy"]
erev_stacked = epot_stacked + df["reversible_energy"]
econrev_stacked = erev_stacked + df["contact_energy"]
edis_stacked = econrev_stacked + df["dissipated_energy"]
econdis_stacked = edis_stacked + df["contact_dissipation"]
einj = (
    df["external_work"] + df["kinetic_energy"].iloc[0] + df["potential_energy"].iloc[0]
)

fig_energy_stacked = go.Figure()
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=ekin_stacked,
        mode="lines",
        name=r"$\mathcal{K}$",
        line=dict(color=colors[0], width=0),
        fill="tozeroy",
        fillcolor=fillcolors[0],
    )
)
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=epot_stacked,
        mode="lines",
        name=r"$\mathcal{U}$",
        line=dict(color=colors[1], width=0),
        fill="tonexty",
        fillcolor=fillcolors[1],
    )
)
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=erev_stacked,
        mode="lines",
        name=r"$\mathcal{R}$",
        line=dict(color=colors[2], width=0),
        fill="tonexty",
        fillcolor=fillcolors[2],
    )
)
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=econrev_stacked,
        mode="lines",
        name=r"$\mathcal{C}_{\rm rev}$",
        line=dict(color=colors[3], width=0),
        fill="tonexty",
        fillcolor=fillcolors[3],
    )
)
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=edis_stacked,
        mode="lines",
        name=r"$\mathcal{G}$",
        line=dict(color=colors[4], width=0),
        fill="tonexty",
        fillcolor=fillcolors[4],
    )
)
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=econdis_stacked,
        mode="lines",
        name=r"$\mathcal{C}_{\rm dis}$",
        line=dict(color=colors[5], width=0),
        fill="tonexty",
        fillcolor=fillcolors[5],
    )
)
fig_energy_stacked.add_trace(
    go.Scatter(
        x=df["time"],
        y=einj,
        mode="lines",
        name=r"$\mathcal{W}^{\rm ext} + \mathcal{E}_0$",
        line=dict(color="red", width=2),
    )
)

fig_energy_stacked.update_layout(
    get_layout(),
    title="Stacked Energy vs. Time",
    xaxis_title="Time (s)",
    yaxis_title=r"$\mathcal{E}$ (J)",
    legend=dict(y=0.9),
)
fig_energy_stacked.show()
if write:
    fig_energy_stacked.write_image(
        f"../../output/{run_id}/stacked_energy_vs_time.pdf",
        width=400,
        height=400,
        scale=2,
    )

# Number of fragments vs. Time
fig_fragments = go.Figure()
fig_fragments.add_trace(
    go.Scatter(
        x=df["time"],
        y=df["nb_fragments"],
        mode="lines",
        name="Number of fragments",
        line=dict(color="black"),
    )
)

fig_fragments.update_layout(
    get_layout(),
    title="Number of fragments vs. Time",
    xaxis_title="Time (s)",
    yaxis_title="Number of fragments",
)
fig_fragments.show()
if write:
    fig_fragments.write_image(
        f"../../output/{run_id}/n_fragments_vs_time.pdf", width=400, height=400, scale=2
    )

In [ ]:
### Plot: fragment mass distribution (CT scan) ###

nbinsx = 100

# Convert final masses to normalized lengths
final_masses = np.asarray(df["fragment_mass"].iloc[-1])
total_mass = np.sum(final_masses)
final_masses_norm = final_masses / total_mass
lmin, lmax = np.min(final_masses_norm), np.max(final_masses_norm)
bins = np.linspace(lmin, lmax, nbinsx + 1)
bin_centers = 0.5 * (bins[:-1] + bins[1:])

# Time axis
y_vals = df["time"].values if "time" in df.columns else df.index.values

# Build Z as absolute fragment counts per time row
Z = np.empty((len(df), nbinsx), dtype=float)
for i, masses in enumerate(df["fragment_mass"].values):
    masses_norm = masses / total_mass
    counts, _ = np.histogram(masses_norm, bins=bins)
    Z[i, :] = counts  # absolute count

# Plot heatmap (CT scan)
fig_length_ct = go.Figure()
fig_length_ct.add_trace(
    go.Heatmap(
        x=bin_centers,
        y=y_vals,
        z=Z,
        colorscale="Greys",
        colorbar=dict(title="Number of fragments"),
    )
)

fig_length_ct.update_layout(
    get_layout(),
    title="Normalized Fragment Mass Distribution (CT scan)",
    xaxis_title="Mass / M̄",
    yaxis_title="Time",
)

fig_length_ct.show()
if write:
    fig_length_ct.write_image(
        f"../../output/{run_id}/fragment_mass_ct_scan_norm.pdf",
        width=400,
        height=400,
        scale=2,
    )

In [ ]:
### Plot: space-time stress diagram ###

write = False

downsample_factor = 1

if quad_coords is not None and stress_matrix is not None:

    fig_stress = go.Figure()
    fig_stress.add_trace(
        go.Heatmap(
            x=quad_coords,
            # Downsampling every N with [::N] if necessary
            y=stress_times[::downsample_factor],
            z=stress_matrix[::downsample_factor, :],  # [::N, :],
            colorscale="RdBu_r",
            zmid=0,
            zmin=-3e8,
            zmax=3e8,
            colorbar=dict(title="Stress [Pa]", thickness=15),
        )
    )

    fig_stress.update_layout(
        get_layout(),
        title=f"Space-Time Stress Diagram ({run_id})",
        xaxis_title="Position [m]",
        yaxis_title="Time [s]",
        yaxis=dict(autorange=True),
    )

    fig_stress.show()

    if write:
        fig_stress.write_image(
            f"../../output/figures/misc/{run_id.split('/')[-1]}_stress_xt_diagram.png",
            width=500,
            height=500,
            scale=4,
        )
else:
    print("Could not plot: Missing quad_coords or stress_matrix from the HDF5 file.")

In [ ]:
### Plot: fragment mass vs. velocity distribution ###

write = False

# Extract data for the last time step
m_final = df["fragment_mass"].iloc[-1]
v_final = df["fragment_velocity"].iloc[-1]

# Create the subplot structure
fig = make_subplots(
    rows=2,
    cols=2,
    column_widths=[0.8, 0.2],
    row_heights=[0.2, 0.8],
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.02,
    horizontal_spacing=0.02,
    subplot_titles=(None, None, None, None),
)

# Add Traces
# Main Scatter (Bottom Left)
fig.add_trace(
    go.Scatter(
        x=m_final,
        y=v_final,
        mode="markers",
        marker=dict(
            color="black", opacity=0.4, size=4, line=dict(width=0.5, color="black")
        ),
        showlegend=False,
    ),
    row=2,
    col=1,
)

# Mass Histogram (Top Left)
fig.add_trace(
    go.Histogram(
        x=m_final,
        nbinsx=100,
        marker_color="black",
        opacity=0.9,
        marker_line_width=0.0,
        marker_line_color="black",
        showlegend=False,
    ),
    row=1,
    col=1,
)

# Velocity Histogram (Bottom Right - note: orientation='h')
fig.add_trace(
    go.Histogram(
        y=v_final,
        nbinsy=100,
        marker_color="black",
        opacity=0.9,
        marker_line_width=0.0,
        marker_line_color="black",
        showlegend=False,
    ),
    row=2,
    col=2,
)

# Apply Layout
fig.update_layout(get_layout())

# Global Axis Styling
axis_style = dict(
    showgrid=True,
    showline=True,
    linewidth=1,
    linecolor="black",
    mirror=True,
    zeroline=False,
    ticks="inside",
    exponentformat="power",
    tickfont=dict(size=12),
)
fig.update_xaxes(axis_style)
fig.update_yaxes(axis_style)
fig.update_xaxes(title_text="Mass (kg)", row=2, col=1)
fig.update_yaxes(title_text="Velocity (m/s)", row=2, col=1)
fig.update_xaxes(title_text="", row=1, col=1)
fig.update_yaxes(title_text="", row=2, col=2)

fig.update_layout(
    title="Fragment Mass vs. Velocity Joint Distribution",
    width=600,
    height=600,  # Square looks better for joint plots
)

fig.show()

if write:
    fig.write_image(f"../../output/{run_id}/joint_mass_velocity.pdf", scale=2)